# Parsing documents with MinerU

This notebook converts raw documents (PDF, DOCX, HTML, images, ...) into Markdown with
MinerU and builds a RAGU knowledge graph from the result.

Prerequisites:

- a running MinerU API server — see [`ragu/preprocessing/README.md`](../ragu/preprocessing/README.md);
- documents to parse in `data/documents/`;
- for the knowledge graph section: `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`.

In [2]:
import os
from pathlib import Path

import requests

from mineru.parser import MinerUApiParser

from ragu.preprocessing.utils import MINERU_SUPPORTED_EXTENSIONS, parse_from_folder

DATA_DIR = Path("data/md/raw")
RESULT_DIR = Path("data/md/result")
FILTERED_RESULT_DIR = Path("data/md/result_pdf")
MINERU_API_URL = os.environ.get("MINERU_API_URL", "http://127.0.0.1:8000")
MINERU_API_KEY = os.environ.get("MINERU_API_KEY", "local")

## Check the MinerU server

In [3]:
response = requests.get(f"{MINERU_API_URL}/v1/health", timeout=5)
response.raise_for_status()
print(response.text)

{"status":"ok","version":"4.0.10","features":{"webhook":false,"output_formats":["markdown","middle_json","structured_content","zip"],"sources":["file_id","url","inline"]}}


## Create a client

`parse_from_folder` takes the MinerU client as an argument, so it is configured here:
server address, key (any placeholder for a self-hosted server) and `tier`, which controls
quality and speed for PDFs and images: `flash`, `basic`, `standard`, `advanced`.

In [4]:
mineru_client = MinerUApiParser(
    api_url=MINERU_API_URL,
    api_key=MINERU_API_KEY,
    tier="standard",
    include_images=True,
    include_model_output=True,
)

## Parse every supported file

With `file_extensions=None` (the default) every format from `MINERU_SUPPORTED_EXTENSIONS`
is parsed. The folder is walked recursively and its structure is mirrored in the output:
`data/documents/a/report.pdf` → `ragu_working_dir/parsed_documents/a/report/`.

Files that fail are logged and skipped.

In [5]:
print(sorted(MINERU_SUPPORTED_EXTENSIONS))

parse_from_folder(mineru_client, DATA_DIR, RESULT_DIR)

['.bmp', '.csv', '.doc', '.docx', '.epub', '.gif', '.htm', '.html', '.jp2', '.jpeg', '.jpg', '.mht', '.mhtml', '.odp', '.ods', '.odt', '.ofd', '.pdf', '.png', '.ppt', '.pptx', '.rtf', '.shtml', '.tiff', '.tsv', '.webp', '.xls', '.xlsx']


100%|██████████| 4/4 [05:22<00:00, 80.57s/it] 


## Parse only selected formats

Extensions are case-insensitive and the leading dot is optional.

In [6]:
parse_from_folder(mineru_client, DATA_DIR, FILTERED_RESULT_DIR, file_extensions={"pdf", ".DOCX"})

100%|██████████| 3/3 [04:49<00:00, 96.54s/it] 


## Inspect the result

In [7]:
markdown_files = sorted(RESULT_DIR.rglob("*.md"))
for path in markdown_files:
    print(path)

if markdown_files:
    print(markdown_files[0].read_text(encoding="utf-8")[:1000])

data\md\result\lithium_ion_batteries\markdown.md
data\md\result\RAGU A Multi-Step GraphRAG Engine with a Compact Domain-Adapted LLM\markdown.md
data\md\result\solar_cells_scan\markdown.md
data\md\result\transistor_history\markdown.md
# Lithium-Ion Batteries: Chemistry, Materials and Performance

*A short technical overview prepared as a sample document for document parsing.*

## 1. History

The first rechargeable lithium cell was built in the 1970s at **Exxon** by **M. Stanley Whittingham**, who used a titanium disulfide cathode. In 1980 **John B. Goodenough** showed that lithium cobalt oxide gives a much higher voltage, and in 1985 **Akira Yoshino** at Asahi Kasei replaced the reactive lithium metal anode with carbon. The three shared the [Nobel Prize in Chemistry 2019](https://www.nobelprize.org/prizes/chemistry/2019/summary/).

*Lithium-ion batteries made portable electronics and modern electric vehicles possible by combining high energy density with long cycle life.*

### 1.1 Key m

## Build a knowledge graph from parsed documents

Parsed Markdown is plain text, so it goes into RAGU like any other corpus. Set
`Settings.language` to the language of your documents.

This is the expensive cell — run it once.

In [ ]:
from ragu import ArtifactsExtractorLLM, KnowledgeGraph, LocalSearchEngine, Settings, SimpleChunker
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.utils.ragu_utils import read_text_from_files

Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/document_parsing_example"

documents = read_text_from_files(RESULT_DIR, file_extensions={".md"})
print(f"{len(documents)} documents")

openai_client = CachedAsyncOpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=openai_client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=openai_client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
)
await knowledge_graph.build_from_docs(documents)

## Ask a question

In [ ]:
search_engine = LocalSearchEngine(llm, knowledge_graph, embedder)

answer = await search_engine.a_query("What are these documents about?")
print(answer.response)

## Clean up

In [ ]:
import shutil

await knowledge_graph.index.close()
for path in (RESULT_DIR, FILTERED_RESULT_DIR, Settings.storage_folder):
    shutil.rmtree(path, ignore_errors=True)
    print(f"removed {path}")